###Ingest transactions file

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, DoubleType

transactions_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_date", DateType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True)
])

In [0]:
transactions_raw_df = (
    spark.read
        .option("header", "true")
        .schema(transactions_schema)
        .csv("abfss://raw@customersegmentationsa.dfs.core.windows.net/transactions.csv")
)

In [0]:
transactions_raw_df.show()

+--------+-----------+----------+----------+-----------+--------+-------+
|order_id|customer_id|order_date|   product|   category|quantity| amount|
+--------+-----------+----------+----------+-----------+--------+-------+
|    T001|       C001|2024-01-01|    Mobile|Electronics|       1|15000.0|
|    T002|       C002|2024-01-02|    Laptop|Electronics|       1|55000.0|
|    T003|       C001|2024-01-05|Headphones|Accessories|       2| 2000.0|
|    T004|       C003|2024-01-10|    Mobile|Electronics|       1|16000.0|
|    T005|       C002|2024-01-15|    Tablet|Electronics|       1|25000.0|
|    T006|       C004|2024-02-01|Smartwatch|Accessories|       1|12000.0|
|    T007|       C005|2024-02-03|    Laptop|Electronics|       1|52000.0|
|    T008|       C002|2024-02-10|    Mobile|Electronics|       1|18000.0|
|    T009|       C001|2024-02-15| PowerBank|Accessories|       2| 3000.0|
|    T010|       C003|2024-02-18|    Mobile|Electronics|       1|15500.0|
|    T011|       C006|2024-02-20|Headp

###Apply data quality & standardisation rules

In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    initcap,
    upper,
    when,
    current_timestamp
)

transactions_processed_df = (
    transactions_raw_df
        # Mandatory fields
        .filter(col("order_id").isNotNull())
        .filter(col("customer_id").isNotNull())
        .filter(col("order_date").isNotNull())

        # Trim & standardize text
        .withColumn("order_id", trim(col("order_id")))
        .withColumn("customer_id", trim(col("customer_id")))
        .withColumn("product", initcap(trim(col("product"))))
        .withColumn("category", upper(trim(col("category"))))

        # Validate numeric fields
        .withColumn(
            "quantity",
            when(col("quantity") <= 0, None)
            .otherwise(col("quantity"))
        )
        .withColumn(
            "amount",
            when(col("amount") <= 0, None)
            .otherwise(col("amount"))
        )

        # Derived metrics
        .withColumn(
            "unit_price",
            when(
                (col("quantity").isNotNull()) & (col("quantity") > 0),
                col("amount") / col("quantity")
            )
        )

        # Audit column
        .withColumn("processed_timestamp", current_timestamp())
)

In [0]:
display(transactions_processed_df)

order_id,customer_id,order_date,product,category,quantity,amount,unit_price,processed_timestamp
T001,C001,2024-01-01,Mobile,ELECTRONICS,1,15000.0,15000.0,2026-03-27T12:15:53.495234Z
T002,C002,2024-01-02,Laptop,ELECTRONICS,1,55000.0,55000.0,2026-03-27T12:15:53.495234Z
T003,C001,2024-01-05,Headphones,ACCESSORIES,2,2000.0,1000.0,2026-03-27T12:15:53.495234Z
T004,C003,2024-01-10,Mobile,ELECTRONICS,1,16000.0,16000.0,2026-03-27T12:15:53.495234Z
T005,C002,2024-01-15,Tablet,ELECTRONICS,1,25000.0,25000.0,2026-03-27T12:15:53.495234Z
T006,C004,2024-02-01,Smartwatch,ACCESSORIES,1,12000.0,12000.0,2026-03-27T12:15:53.495234Z
T007,C005,2024-02-03,Laptop,ELECTRONICS,1,52000.0,52000.0,2026-03-27T12:15:53.495234Z
T008,C002,2024-02-10,Mobile,ELECTRONICS,1,18000.0,18000.0,2026-03-27T12:15:53.495234Z
T009,C001,2024-02-15,Powerbank,ACCESSORIES,2,3000.0,1500.0,2026-03-27T12:15:53.495234Z
T010,C003,2024-02-18,Mobile,ELECTRONICS,1,15500.0,15500.0,2026-03-27T12:15:53.495234Z


###Save the output as delta table to processed container

In [0]:
transactions_processed_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("processed_db.transactions")